<a href="https://colab.research.google.com/github/juliaR0/SF_10_new/blob/main/notebooks/Prototype_2_1_112326.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prototype 2.1 — Patient in, recommendation out (wired, not accurate)

*Code in this notebook was drafted with Claude (Anthropic) and run, debugged and adapted by me. Prompts are in my AI log.*

**Goal:** take a real patient's language history (LEX-BADAT) → build 3 twins → damage them to a chosen severity → simulate English / Spanish / mixed / sham therapy → score (strict) → rank the options → print a report card. Every step uses a simple written rule; accuracy is not the point yet.

**Builds on:** Prototype 1.2 (strict scoring), the Bilingual Dual-path model (Tsoukala et al., 2019), BiLex's idea of patients as language-exposure histories with simulated therapy (Kiran, Grasemann et al.), LEX-BADAT profiles (Marte et al., 2022, CC BY).

**Rules are assumptions.** Each one is marked `# RULE` in the code and gets a row in the decision log.

**Run order:** Section 1 always. Every model run skips itself if its results folder already exists.

# 1. Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/drive/MyDrive/dual_path'):
    !git clone https://github.com/xtsoukala/dual_path.git /content/drive/MyDrive/dual_path
%cd /content/drive/MyDrive/dual_path

Mounted at /content/drive
/content/drive/MyDrive/dual_path


In [4]:
!sed 's/==.*//' requirements.txt > requirements_unpinned.txt
!pip install -q -r requirements_unpinned.txt

In [5]:
!printf "simulations/\ndata_private/\n*.cha\nchen_*.csv\nLEX-BADAT*\n" >> .gitignore
!cat .gitignore

__pycache__/
simulations/
env/
.DS_Store
simulations/
data_private/
*.cha
chen_*.csv
LEX-BADAT*
simulations/
data_private/
*.cha
chen_*.csv
LEX-BADAT*


In [6]:
import sys, os, copy, json, glob, math, pickle, shutil, lz4.frame, torch
import pandas as pd
sys.path.insert(0, 'src')
pd.set_option('display.max_rows', 200, 'display.max_colwidth', 100)

def dp(args, resdir):
    """Run the model unless simulations/<resdir> already exists."""
    if os.path.exists(f'simulations/{resdir}'):
        print('skip (already exists):', resdir); return
    cmd = f'python src/start_dual_path.py {args} --resdir {resdir}'
    print(cmd); get_ipython().system(cmd)

def lang_of(msg):
    return 'es' if 'TARGET-LANG=es' in str(msg) else 'en'

def structure(msg):
    msg = str(msg)
    if 'RECIPIENT=' in msg: return '3 ditransitive'
    if 'PATIENT=' in msg:   return '2 transitive'
    if 'ATTR=' in msg:      return 'copular'
    return '1 intransitive'

LEX = pd.read_csv('data/code-switching/lexicon.csv')
ES_ONLY = set(LEX.morpheme_es.dropna()) - set(LEX.morpheme_en.dropna())
EN_ONLY = set(LEX.morpheme_en.dropna()) - set(LEX.morpheme_es.dropna())

def load_test(path, epoch=None):
    """Read test.csv; add m (repo meaning label), tlang, struct, strict (right meaning AND target language only)."""
    d = pd.read_csv(path)
    if epoch is not None: d = d[d.epoch == epoch].copy()
    d['m'] = d['meaning'].astype(str).str.strip()
    d['tlang'] = d.message.apply(lang_of)
    d['struct'] = d.message.apply(structure)
    def ok(r):
        words = set(str(r.produced_sentence).split())
        wrong = ES_ONLY if r.tlang == 'en' else EN_ONLY
        return r.m == 'True' and not (words & wrong)
    d['strict'] = d.apply(ok, axis=1)
    return d

def pct(mask): return round(100 * mask.mean(), 1)

def load_weights(path):
    with lz4.frame.open(path, 'rb') as f: return pickle.load(f)

def lesion(layers, layer_name, fraction, seed=0):
    """Damaged COPY: zero a random `fraction` of layer_name's incoming weights."""
    d = copy.deepcopy(layers)
    W = d[layer_name].in_weights
    g = torch.Generator().manual_seed(seed)
    mask = torch.rand(W.shape, generator=g) < fraction
    d[layer_name].in_weights = W.masked_fill(mask, 0.0)
    return d

def save_layers(layers, folder, epoch):
    """Save where the repo looks: folder/1/weights and folder/weights."""
    for path in [f'{folder}/1/weights', f'{folder}/weights']:
        os.makedirs(path, exist_ok=True)
        with lz4.frame.open(f'{path}/w{epoch}.lz4', 'wb') as f:
            pickle.dump(layers, f, protocol=-1)

def build_input(folder, src_run, src_sim, keep=None):
    """--input folder: folder/input (shared files) + folder/1 (training.in = kept lines, test.in)."""
    os.makedirs(f'{folder}/input', exist_ok=True); os.makedirs(f'{folder}/1', exist_ok=True)
    for f in os.listdir(f'{src_run}/input'):
        p = f'{src_run}/input/{f}'
        if os.path.isfile(p): shutil.copyfile(p, f'{folder}/input/{f}')
    lines = open(f'{src_run}/{src_sim}/training.in').read().splitlines()
    if keep is not None: lines = keep(lines)
    open(f'{folder}/1/training.in', 'w').write('\n'.join(lines) + '\n')
    shutil.copyfile(f'{src_run}/{src_sim}/test.in', f'{folder}/1/test.in')
    return folder

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


# 2. Step 1 — Patients (LEX-BADAT)

LEX-BADAT stores L2 age of acquisition ×(−1); I flip it back to a normal age. Use = share of daily use before the stroke. **Assumption:** L1 = Spanish (true for 72 of 85 patients; per-person L1 not in the file).

*Data: Marte et al. (2022), LEX-BADAT, Frontiers in Psychology 13:875928, CC BY. Raw files stay in `data_private/` (not pushed to GitHub).*

In [7]:
os.makedirs('analysis', exist_ok=True)
lux = pd.read_csv('data_private/LEX-BADAT_BPWA_LUQ.txt', sep='\t')
lux['AoA'] = -lux.L2AoA

CHOSEN = ['P77', 'P14']        # <-- my choice; reasons in the decision log
patients = lux[lux.BPWA.isin(CHOSEN)][['BPWA', 'AoA', 'L1PreUse', 'L2PreUse', 'L1PreLAR', 'L2PreLAR']].copy()
patients.to_csv('analysis/p21_patients.csv', index=False)
patients

,BPWA,AoA,L1PreUse,L2PreUse,L1PreLAR,L2PreLAR
13,P14,15,0.926471,0.073529,1.0,0.8
76,P77,0,0.435484,0.564516,1.0,1.0


# 3. Step 2 — Rule 1: profile → twin settings

- `# RULE` **AoA → epoch English starts:** `start = min(10, round(AoA / 2))` (AoA 0 → epoch 0, AoA 15 → epoch 8, AoA 20+ → epoch 10). One epoch ≈ two years of childhood; capped so late learners still get English training.
- `# RULE` **Use → language mix:** English share of training sentences = pre-stroke English use (L2PreUse), kept between 0.1 and 0.9 so both languages are learned.
- `# RULE` **Amount of English training:** 15 epochs after English starts (lesson from 1.2: 10 was too few).

In [8]:
L2_EPOCHS = 15                                            # RULE

def profile_to_settings(row):
    start = min(10, round(row.AoA / 2))                   # RULE
    mix = min(0.9, max(0.1, float(row.L2PreUse)))         # RULE
    return {'id': row.BPWA, 'start': int(start), 'mix': round(mix, 2), 'final_epoch': L2_EPOCHS}

SETTINGS = {r.BPWA: profile_to_settings(r) for r in patients.itertuples()}
pd.DataFrame(SETTINGS.values())

,id,start,mix,final_epoch
0,P14,8,0.10,15
1,P77,0,0.56,15


# 4. Step 3 — Train 3 twins per patient

Twins share the settings and differ in random start (twin number = seed). Early learners: one run. Late learners: Spanish only until `start`, then load and add English.

In [9]:
N_TWINS = 3

def train_twins(s):
    pid, start, mix = s['id'], s['start'], s['mix']
    base = '--cs --lang es en --norandomization'
    if start == 0:
        run = f'p21_{pid}_twins'
        dp(f'--sim {N_TWINS} --epochs {L2_EPOCHS} {base} --l2-decimal-fraction {mix}', run)
    else:
        step1 = f'p21_{pid}_step1'
        dp(f'--sim {N_TWINS} --epochs {start} {base} --l2-decimal-fraction 0', step1)
        os.makedirs(f'simulations/{step1}/weights', exist_ok=True)           # startup quirk
        shutil.copyfile(f'simulations/{step1}/1/weights/w{start}.lz4', f'simulations/{step1}/weights/w{start}.lz4')
        run = f'p21_{pid}_twins'
        dp(f'--sim {N_TWINS} --epochs {L2_EPOCHS} {base} --l2-decimal-fraction {mix} --sw simulations/{step1} --swe {start}', run)
    s['run'] = f'simulations/{run}'
    return s

for pid in SETTINGS:
    train_twins(SETTINGS[pid])

skip (already exists): p21_P14_step1
skip (already exists): p21_P14_twins
skip (already exists): p21_P77_twins


In [10]:
# Healthy check: each twin's strict accuracy by language at the end of training
rows = []
for pid, s in SETTINGS.items():
    for k in range(1, N_TWINS + 1):
        d = load_test(f"{s['run']}/{k}/test.csv", epoch=s['final_epoch'])
        for tl, g in d.groupby('tlang'):
            rows.append({'patient': pid, 'twin': k, 'lang': tl, 'strict_%': pct(g.strict), 'switch_%': pct(g.is_code_switched.astype(str).str.strip() == 'True')})
healthy = pd.DataFrame(rows)
healthy.pivot_table(index=['patient', 'twin'], columns='lang', values='strict_%')

lang            en    es
patient twin            
P14     1      8.0  96.7
        2      6.0  98.7
        3      4.0  98.7
P77     1     33.9  86.8
        2     66.8  80.9
        3     83.9  80.0

TPU CHECK

In [1]:
import os
print(os.cpu_count())

24


# 5. Step 4 — Rule 2: severity → damage (calibration by simulation)

- `# RULE` **Damage site:** grammar path (hidden → pred_compress), because these are modeled as agrammatic patients.
- `# RULE` **Severity target:** without per-patient WAB, the damaged twin should keep about **60% of its healthy strict accuracy** (moderate aphasia; Chen's patients were mild–moderate). Pick the damage level closest to that.

LEVELS = [0.0, 0.2, 0.3, 0.4, 0.5, 0.6]
TARGET_RATIO = 0.60                                       # RULE

def twin_args(s, k):
    w = load_weights(f"{s['run']}/{k}/weights/w{s['final_epoch']}.lz4")
    return w, f"--norandomization --sim 1 --hidden {w['hidden'].size} --compress {w['compress'].size} --fw 15 --cs --lang es en"

def eval_lesion(s, k, frac):
    pid, E = s['id'], s['final_epoch']
    tag = f"p21_{pid}_t{k}_les{int(frac*100):02d}"
    w, args = twin_args(s, k)
    folder = f'simulations/{tag}_w'
    if not os.path.exists(f'{folder}/weights/w{E}.lz4'):
        save_layers(lesion(w, 'pred_compress', frac, seed=k), folder, E)
    inp = build_input(f'simulations/{tag}_in', s['run'], k)
    dp(f'{args} --epochs 1 --sw {folder} --swe {E} --input {inp}', f'{tag}_eval')
    return load_test(f'simulations/{tag}_eval/1/test.csv', epoch=0)

calib = []
for pid, s in SETTINGS.items():
    for k in range(1, N_TWINS + 1):
        for frac in LEVELS:
            d = eval_lesion(s, k, frac)
            calib.append({'patient': pid, 'twin': k, 'damage': frac, 'strict_%': pct(d.strict)})
calib = pd.DataFrame(calib)
calib['healthy_%'] = calib.groupby(['patient', 'twin'])['strict_%'].transform(lambda x: x.iloc[0])
calib['ratio'] = (calib['strict_%'] / calib['healthy_%']).round(2)

CHOSEN_DAMAGE = {}
for (pid, k), g in calib.groupby(['patient', 'twin']):
    best = g.iloc[(g.ratio - TARGET_RATIO).abs().argsort()].iloc[0]
    CHOSEN_DAMAGE[(pid, k)] = best.damage
calib.pivot_table(index=['patient', 'twin'], columns='damage', values='ratio'), CHOSEN_DAMAGE

In [10]:
import subprocess, concurrent.futures as cf
WORKERS = 18                                              # 24 CPUs available; 36 runs = 2 rounds

def run_many(jobs):
    """Run many model runs at once. jobs = list of (args, resdir). Skips ones already done."""
    todo = [(a, r) for a, r in jobs if not os.path.exists(f'simulations/{r}')]
    print(len(jobs) - len(todo), 'already done;', len(todo), 'to run on', WORKERS, 'workers')
    env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')   # one core per run
    def one(job):
        a, r = job
        p = subprocess.run(f'python src/start_dual_path.py {a} --resdir {r}',
                           shell=True, capture_output=True, text=True, env=env)
        return r, p.returncode, p.stderr[-400:]
    with cf.ThreadPoolExecutor(WORKERS) as ex:
        for r, code, err in ex.map(one, todo):
            print('ok    ' if code == 0 else 'FAILED', r, '' if code == 0 else err)

LEVELS = [0.0, 0.2, 0.3, 0.4, 0.5, 0.6]
TARGET_RATIO = 0.60                                       # RULE

def twin_args(s, k):
    w = load_weights(f"{s['run']}/{k}/weights/w{s['final_epoch']}.lz4")
    return w, f"--norandomization --sim 1 --hidden {w['hidden'].size} --compress {w['compress'].size} --fw 15 --cs --lang es en"

def prep_lesion(s, k, frac):
    pid, E = s['id'], s['final_epoch']
    tag = f"p21_{pid}_t{k}_les{int(frac*100):02d}"
    w, args = twin_args(s, k)
    folder = f'simulations/{tag}_w'
    if not os.path.exists(f'{folder}/weights/w{E}.lz4'):
        save_layers(lesion(w, 'pred_compress', frac, seed=k), folder, E)
    inp = build_input(f'simulations/{tag}_in', s['run'], k)
    return (f'{args} --epochs 1 --sw {folder} --swe {E} --input {inp}', f'{tag}_eval')

jobs = [prep_lesion(s, k, f) for s in SETTINGS.values() for k in range(1, N_TWINS + 1) for f in LEVELS]
run_many(jobs)

calib = []
for pid, s in SETTINGS.items():
    for k in range(1, N_TWINS + 1):
        for frac in LEVELS:
            d = load_test(f"simulations/p21_{pid}_t{k}_les{int(frac*100):02d}_eval/1/test.csv", epoch=0)
            calib.append({'patient': pid, 'twin': k, 'damage': frac, 'strict_%': pct(d.strict)})
calib = pd.DataFrame(calib)
calib['healthy_%'] = calib.groupby(['patient', 'twin'])['strict_%'].transform(lambda x: x.iloc[0])
calib['ratio'] = (calib['strict_%'] / calib['healthy_%']).round(2)

CHOSEN_DAMAGE = {}
for (pid, k), g in calib.groupby(['patient', 'twin']):
    CHOSEN_DAMAGE[(pid, k)] = g.iloc[(g.ratio - TARGET_RATIO).abs().argsort()].iloc[0].damage
calib.pivot_table(index=['patient', 'twin'], columns='damage', values='ratio'), CHOSEN_DAMAGE

0 already done; 36 to run on 18 workers
ok     p21_P14_t1_les00_eval 
ok     p21_P14_t1_les20_eval 
ok     p21_P14_t1_les30_eval 
ok     p21_P14_t1_les40_eval 
ok     p21_P14_t1_les50_eval 
ok     p21_P14_t1_les60_eval 
ok     p21_P14_t2_les00_eval 
ok     p21_P14_t2_les20_eval 
ok     p21_P14_t2_les30_eval 
ok     p21_P14_t2_les40_eval 
ok     p21_P14_t2_les50_eval 
ok     p21_P14_t2_les60_eval 
ok     p21_P14_t3_les00_eval 
ok     p21_P14_t3_les20_eval 
ok     p21_P14_t3_les30_eval 
ok     p21_P14_t3_les40_eval 
ok     p21_P14_t3_les50_eval 
ok     p21_P14_t3_les60_eval 
ok     p21_P77_t1_les00_eval 
ok     p21_P77_t1_les20_eval 
ok     p21_P77_t1_les30_eval 
ok     p21_P77_t1_les40_eval 
ok     p21_P77_t1_les50_eval 
ok     p21_P77_t1_les60_eval 
ok     p21_P77_t2_les00_eval 
ok     p21_P77_t2_les20_eval 
ok     p21_P77_t2_les30_eval 
ok     p21_P77_t2_les40_eval 
ok     p21_P77_t2_les50_eval 
ok     p21_P77_t2_les60_eval 
ok     p21_P77_t3_les00_eval 
ok     p21_P77_t3_les20_eval 


(damage        0.0   0.2   0.3   0.4   0.5   0.6
 patient twin                                   
 P14     1     1.0  1.01  1.01  0.99  0.94  0.80
         2     1.0  0.95  0.92  0.89  0.77  0.59
         3     1.0  0.96  0.94  0.91  0.70  0.47
 P77     1     1.0  0.89  0.85  0.70  0.66  0.52
         2     1.0  0.97  1.03  0.93  0.87  0.70
         3     1.0  0.98  0.95  0.94  0.83  0.65,
 {('P14', np.int64(1)): np.float64(0.6),
  ('P14', np.int64(2)): np.float64(0.6),
  ('P14', np.int64(3)): np.float64(0.5),
  ('P77', np.int64(1)): np.float64(0.5),
  ('P77', np.int64(2)): np.float64(0.6),
  ('P77', np.int64(3)): np.float64(0.6)})

# 6. Step 5 — Therapy options

- `# RULE` **Options:** English ditransitives, Spanish ditransitives, **mixed** (half English, half Spanish ditransitives), **sham** (intransitives, half each language).
- `# RULE` **Dose:** same number of sentences for every option (= the smaller of the twin's English and Spanish ditransitive counts), 5 sessions, learning rate 0.02.

SESSIONS = 5
OPTIONS = ['EN', 'ES', 'MIXED', 'SHAM']

def therapy_lines(lines, option):
    ditr = lambda lg: [l for l in lines if 'RECIPIENT=' in l and f'TARGET-LANG={lg}' in l]
    intr = lambda lg: [l for l in lines if structure(l.split('##')[-1]) == '1 intransitive' and f'TARGET-LANG={lg}' in l]
    n = min(len(ditr('en')), len(ditr('es')))
    if option == 'EN':    return ditr('en')[:n]
    if option == 'ES':    return ditr('es')[:n]
    if option == 'MIXED': return ditr('en')[:n//2] + ditr('es')[:n - n//2]
    if option == 'SHAM':  return intr('en')[:n//2] + intr('es')[:n - n//2]

def run_therapy(s, k, option):
    pid, E = s['id'], s['final_epoch']
    frac = CHOSEN_DAMAGE[(pid, k)]
    _, args = twin_args(s, k)
    les = f"simulations/p21_{pid}_t{k}_les{int(frac*100):02d}_w"
    tag = f"p21_{pid}_t{k}_th{option}"
    inp = build_input(f'simulations/{tag}_in', s['run'], k, keep=lambda L: therapy_lines(L, option))
    dp(f'{args} --epochs {SESSIONS} --lrate 0.02 --nodlr --sw {les} --swe {E} --input {inp}', f'{tag}_run')
    return f'simulations/{tag}_run/1/test.csv'

THERAPY = {}
for pid, s in SETTINGS.items():
    for k in range(1, N_TWINS + 1):
        for opt in OPTIONS:
            THERAPY[(pid, k, opt)] = run_therapy(s, k, opt)
print(len(THERAPY), 'therapy runs')

In [11]:
SESSIONS = 5
OPTIONS = ['EN', 'ES', 'MIXED', 'SHAM']                  # RULE

def therapy_lines(lines, option):                        # RULE: equal dose for every option
    ditr = lambda lg: [l for l in lines if 'RECIPIENT=' in l and f'TARGET-LANG={lg}' in l]
    intr = lambda lg: [l for l in lines if structure(l.split('##')[-1]) == '1 intransitive' and f'TARGET-LANG={lg}' in l]
    n = min(len(ditr('en')), len(ditr('es')))
    if option == 'EN':    return ditr('en')[:n]
    if option == 'ES':    return ditr('es')[:n]
    if option == 'MIXED': return ditr('en')[:n//2] + ditr('es')[:n - n//2]
    if option == 'SHAM':  return intr('en')[:n//2] + intr('es')[:n - n//2]

def prep_therapy(s, k, option):
    pid, E = s['id'], s['final_epoch']
    frac = CHOSEN_DAMAGE[(pid, k)]
    _, args = twin_args(s, k)
    les = f"simulations/p21_{pid}_t{k}_les{int(frac*100):02d}_w"
    tag = f"p21_{pid}_t{k}_th{option}"
    inp = build_input(f'simulations/{tag}_in', s['run'], k, keep=lambda L: therapy_lines(L, option))
    return (f'{args} --epochs {SESSIONS} --lrate 0.02 --nodlr --sw {les} --swe {E} --input {inp}', f'{tag}_run')

THERAPY, jobs = {}, []
for pid, s in SETTINGS.items():
    for k in range(1, N_TWINS + 1):
        for opt in OPTIONS:
            a, r = prep_therapy(s, k, opt)
            jobs.append((a, r)); THERAPY[(pid, k, opt)] = f'simulations/{r}/1/test.csv'
run_many(jobs)

0 already done; 24 to run on 18 workers
ok     p21_P14_t1_thEN_run 
ok     p21_P14_t1_thES_run 
ok     p21_P14_t1_thMIXED_run 
ok     p21_P14_t1_thSHAM_run 
ok     p21_P14_t2_thEN_run 
ok     p21_P14_t2_thES_run 
ok     p21_P14_t2_thMIXED_run 
ok     p21_P14_t2_thSHAM_run 
ok     p21_P14_t3_thEN_run 
ok     p21_P14_t3_thES_run 
ok     p21_P14_t3_thMIXED_run 
ok     p21_P14_t3_thSHAM_run 
ok     p21_P77_t1_thEN_run 
ok     p21_P77_t1_thES_run 
ok     p21_P77_t1_thMIXED_run 
ok     p21_P77_t1_thSHAM_run 
ok     p21_P77_t2_thEN_run 
ok     p21_P77_t2_thES_run 
ok     p21_P77_t2_thMIXED_run 
ok     p21_P77_t2_thSHAM_run 
ok     p21_P77_t3_thEN_run 
ok     p21_P77_t3_thES_run 
ok     p21_P77_t3_thMIXED_run 
ok     p21_P77_t3_thSHAM_run 


# 7. Step 6 — Scores (strict), with spread across twins

- **Gain** = strict accuracy on ditransitives after 5 sessions minus before (epoch 0 of the same run), per language.
- **Forgetting** = average drop on the untreated structures (intransitive, transitive, copular), both languages; only drops count.

In [12]:
def outcome(path):
    d0, d5 = load_test(path, epoch=0), load_test(path, epoch=SESSIONS)
    out = {}
    for lg in ['en', 'es']:
        a = d0[(d0.struct == '3 ditransitive') & (d0.tlang == lg)].strict.mean()
        b = d5[(d5.struct == '3 ditransitive') & (d5.tlang == lg)].strict.mean()
        out[f'gain_{lg}'] = round(100 * (b - a), 1)
    drops = []
    for st in ['1 intransitive', '2 transitive', 'copular']:
        for lg in ['en', 'es']:
            a = d0[(d0.struct == st) & (d0.tlang == lg)].strict.mean()
            b = d5[(d5.struct == st) & (d5.tlang == lg)].strict.mean()
            drops.append(max(0.0, 100 * (a - b)))
    out['forgetting'] = round(sum(drops) / len(drops), 1)
    return out

res = pd.DataFrame([{'patient': pid, 'twin': k, 'option': opt, **outcome(p)} for (pid, k, opt), p in THERAPY.items()])
# RULE: benefit = average ditransitive gain over both languages minus forgetting
res['benefit'] = ((res.gain_en + res.gain_es) / 2 - res.forgetting).round(1)
res.to_csv('analysis/p21_therapy_outcomes.csv', index=False)
res.groupby(['patient', 'option'])[['gain_en', 'gain_es', 'forgetting', 'benefit']].agg(['mean', 'min', 'max']).round(1)

gain_en             gain_es             forgetting             \
                  mean   min   max    mean   min   max       mean  min   max   
patient option                                                                 
P14     EN         0.0   0.0   0.0   -12.9 -36.2   6.1        7.5  2.7  14.5   
        ES         0.0   0.0   0.0    53.8  23.4  78.7        0.0  0.0   0.0   
        MIXED      0.0   0.0   0.0    43.5   8.5  87.2        0.4  0.0   0.8   
        SHAM       0.0   0.0   0.0    21.5  12.8  36.7        0.0  0.0   0.0   
P77     EN        46.7  10.0  71.4    -3.1  -5.0   0.0       12.2  6.7  19.5   
        ES        -6.8 -16.7   0.0    49.3  35.0  65.2       14.7  9.5  21.4   
        MIXED     45.3  20.0  60.7    34.2   0.0  73.9        1.1  0.0   2.4   
        SHAM      16.2 -13.8  35.7     8.8   0.0  21.7        0.6  0.0   1.0   

               benefit              
                  mean   min   max  
patient option                      
P14     EN       -13.9 -20.8  -2.2  
        ES        26.9  11.7  39.4  
        MIXED     21.3   4.2  42.8  
        SHAM      10.7   6.4  18.4  
P77     EN         9.6 -17.0  23.1  
        ES         6.5 -12.2  17.5  
        MIXED     38.7   9.2  67.3  
        SHAM      11.9  -5.5  28.7

# 8. Step 7 — recommend()

`# RULE` Rank options by mean benefit across twins. If the best option's worst twin is not better than the runner-up's best twin (ranges overlap), the honest answer is **"no clear winner"**. Sham is the baseline: an option only counts if it beats sham.

In [13]:
def recommend(pid):
    r = res[res.patient == pid].groupby('option').benefit.agg(['mean', 'min', 'max']).sort_values('mean', ascending=False)
    real = r.drop(index='SHAM')
    best, second = real.index[0], real.index[1]
    beats_sham = real.loc[best, 'min'] > r.loc['SHAM', 'max']
    clear = real.loc[best, 'min'] > real.loc[second, 'max']
    if not beats_sham:
        verdict = 'no option clearly beats sham therapy'
    elif clear:
        verdict = f'{best} (clearly ahead of {second})'
    else:
        verdict = f'no clear winner ({best} and {second} overlap)'
    return r, verdict

# 9. Step 8 — Report cards

In [14]:
for pid in SETTINGS:
    p = patients[patients.BPWA == pid].iloc[0]
    s = SETTINGS[pid]
    table, verdict = recommend(pid)
    print('=' * 70)
    print(f'PATIENT {pid}  (LEX-BADAT)   English AoA {p.AoA:.0f}, pre-stroke English use {p.L2PreUse:.0%}')
    print(f'Twins: {N_TWINS}; English from epoch {s["start"]}, English share {s["mix"]}; damage per twin', {k: CHOSEN_DAMAGE[(pid, k)] for k in range(1, N_TWINS + 1)})
    print('Predicted benefit by therapy option (mean, worst twin, best twin):')
    print(table.round(1).to_string())
    print('RECOMMENDATION:', verdict)
    print('Caveat: prototype; 3 twins, toy language, assumed rules; a hypothesis, not clinical advice.')

PATIENT P14  (LEX-BADAT)   English AoA 15, pre-stroke English use 7%
Twins: 3; English from epoch 8, English share 0.1; damage per twin {1: np.float64(0.6), 2: np.float64(0.6), 3: np.float64(0.5)}
Predicted benefit by therapy option (mean, worst twin, best twin):
        mean   min   max
option                  
ES      26.9  11.7  39.4
MIXED   21.3   4.2  42.8
SHAM    10.7   6.4  18.4
EN     -13.9 -20.8  -2.2
RECOMMENDATION: no option clearly beats sham therapy
Caveat: prototype; 3 twins, toy language, assumed rules; a hypothesis, not clinical advice.
PATIENT P77  (LEX-BADAT)   English AoA 0, pre-stroke English use 56%
Twins: 3; English from epoch 0, English share 0.56; damage per twin {1: np.float64(0.5), 2: np.float64(0.6), 3: np.float64(0.6)}
Predicted benefit by therapy option (mean, worst twin, best twin):
        mean   min   max
option                  
MIXED   38.7   9.2  67.3
SHAM    11.9  -5.5  28.7
EN       9.6 -17.0  23.1
ES       6.5 -12.2  17.5
RECOMMENDATION: no option 

# 10. Prototype 2.1 summary *(fill in after running)*

**What worked:**

**What broke / surprised me:**

**Decision log rows added:** AoA → start epoch · use → mix · 15 English epochs · severity target 60% · grammar-path site · therapy options and dose · benefit formula · "no clear winner" rule · patient choice

**Next (2.2):** structure vs language-choice scores, tense scorer, compare with Chen.